# Package import

In [14]:
import os
import yaml
import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import pandas as pd 
import io
from ics import Calendar, Event
from datetime import datetime
from zoneinfo import ZoneInfo

# Apollo Scraper

In [3]:
with open('config.yaml','r', encoding="utf-8") as f:
    config=yaml.safe_load(f)
username=config["credentials"]["username"]
password=config["credentials"]["password"]
credentials=HTTPBasicAuth(username, password)

In [4]:
group_id = int(input("Enter group id: "))
# 252681
url = f"https://planzajec.uek.krakow.pl/index.php?typ=G&id={group_id}&okres=2"
response = requests.get(url,auth = credentials)
response.encoding = 'utf-8'
print(response.status_code)

200


In [5]:
page_dom=BeautifulSoup(response.text, "html.parser")

In [6]:
group=page_dom.select_one("div.grupa").get_text(strip=True)
print(group)
# lectures=page_dom.find_all("th")
# print(lectures)

ZICSS1-1211


# Not needed

-----
----

In [ ]:
headers=page_dom.find_all("th")
headers_points=[x.text for x in headers]

rows=page_dom.find_all("tr")[1:]
rows_points=[]
for row in rows:
    elements=row.find_all("td")
    if len(elements)>2:
        rows_points.append([x.text  for x in elements])
print(headers_points)
print(rows_points)
dp=pd.DataFrame(rows_points,columns=headers_points)
print(dp)
dp.to_csv("aa.csv", index=False, encoding='utf-8')

---
---

In [7]:
classes_tag=page_dom.select_one("table")
with open("temp.html", 'w', encoding="utf-8") as f:
    f.write(classes_tag.prettify())
classes=pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")
print(classes)


         Termin          Dzień, godzina  \
0    2026-02-23  Pn 11:30 - 13:00 (2g.)   
1    2026-02-23  Pn 13:15 - 15:45 (3g.)   
2    2026-02-23  Pn 13:15 - 15:45 (3g.)   
3    2026-02-23  Pn 18:30 - 20:00 (2g.)   
4    2026-02-24  Wt 09:45 - 11:15 (2g.)   
..          ...                     ...   
295  2026-06-11  Cz 16:45 - 17:30 (1g.)   
296  2026-06-11  Cz 18:30 - 20:00 (2g.)   
297  2026-06-11  Cz 18:30 - 20:00 (2g.)   
298  2026-06-17  Śr 10:30 - 13:00 (3g.)   
299  2026-09-03  Cz 10:30 - 13:00 (3g.)   

                                   Przedmiot        Typ  \
0                         Foreign Language I   lektorat   
1                     Computer Programming 2  ćwiczenia   
2                                    CS gr 1    CS gr 1   
3                                        NaN   lektorat   
4                 Probability and Statistics     wykład   
..                                       ...        ...   
295                      Information Systems  ćwiczenia   
296  Operat

In [8]:
classes = classes.loc[classes["Typ"].isin(["ćwiczenia", "wykład", "egzamin"])]

In [9]:
classes[["Day","Start time", "Hypnen", "End time", "Duration"]]=classes["Dzień, godzina"].str.split(" ", expand=True)
classes["Duration"] = classes["Duration"].str.extract(r"(\d+)").astype(int)
classes=classes.drop(columns=["Dzień, godzina","Hypnen"])

In [10]:

classes["Sala"] = classes["Sala"].str.replace(r'Win.*', '', regex=True)

In [11]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")


In [13]:
classes.to_csv(f"schedules/{group}.csv")

In [41]:
c=Calendar()

# Set the timezone to ensure times align correctly locally
timezone = ZoneInfo("Europe/Warsaw")
uniqueType = classes["Typ"].unique()
calendars = {} 
for typ_name in uniqueType:
    calendars[typ_name] = Calendar()

# Iterate through the DataFrame rows
for index, row in classes.iterrows():
    e = Event()
    
    # Format the Event Title: e.g., "Computer Programming 2 (ćwiczenia)"
    e.name = f"{row['Przedmiot']} ({row['Typ']})"
    
    # Combine the 'Termin' and 'Start/End time' columns into datetime objects
    start_string = f"{row['Termin']} {row['Start time']}"
    end_string = f"{row['Termin']} {row['End time']}"
    
    # Parse the strings into timezone-aware datetime objects
    e.begin = datetime.strptime(start_string, "%Y-%m-%d %H:%M").replace(tzinfo=timezone)
    e.end = datetime.strptime(end_string, "%Y-%m-%d %H:%M").replace(tzinfo=timezone)
  
    # Map the location and description
    e.location = row['Sala']
    e.description = f"Prowadzący: {row['Nauczyciel']}"



    typ_name=row["Typ"]
    calendars[typ_name].events.add(e)
    # Add the configured event to the calendar
    c.events.add(e)


# Save the calendar to an .ics file
output_filename = 'university_schedule.ics'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.writelines(c.serialize_iter())

for type_name, value in calendars.items():
    filename=f"university_schedule_{type_name}.ics"
    with open(filename,"w",encoding="utf-8") as f:
        f.writelines(value.serialize_iter())


print(f"Successfully generated {output_filename} with {len(classes)} events!")

{'ćwiczenia': <Calendar with 0 event and 0 todo>, 'wykład': <Calendar with 0 event and 0 todo>, 'egzamin': <Calendar with 0 event and 0 todo>}
{'ćwiczenia': <Calendar with 59 events and 0 todo>, 'wykład': <Calendar with 62 events and 0 todo>, 'egzamin': <Calendar with 2 events and 0 todo>}
Successfully generated university_schedule.ics with 123 events!
